# 01 — Threading et GIL

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer et gérer des threads avec `threading.Thread` ;
- comprendre le **Global Interpreter Lock (GIL)** et ses conséquences sur la performance ;
- utiliser `Lock`, `RLock` et `Event` pour synchroniser des threads ;
- identifier les cas d'usage adaptés au threading (I/O-bound) ;
- éviter les pièges classiques : race conditions, deadlocks, données partagées.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- le modèle objet complet (héritage, MRO, `super`, méthodes spéciales) ;
- les décorateurs de fonction et de classe ;
- les gestionnaires de contexte (`with`) ;
- les fonctions lambda, closures et higher-order functions ;
- la métaprogrammation (descripteurs, `__init_subclass__`, métaclasses, ABC/Protocol) ;
- `slots`, `weakref`, le ramasse-miettes, le bytecode (`dis`).

Notions que nous allons **introduire** ici :

- le module `threading` et la classe `Thread` ;
- le GIL, son impact, et les stratégies pour travailler avec ;
- les primitives de synchronisation : `Lock`, `RLock`, `Event`.

## Plan

1. Qu'est-ce qu'un thread ?
2. Créer un thread avec `threading.Thread`
3. Threads démons vs non-démons
4. Le GIL — Global Interpreter Lock
5. `Lock` — exclusion mutuelle
6. `RLock` — verrou réentrant
7. `Event` — signalisation entre threads
8. `local()` — données thread-local
9. Race conditions et débogage
10. Synthèse
11. Exercices

---

## 1. Qu'est-ce qu'un thread ?

Un **thread** (fil d'exécution) est une unité d'exécution au sein d'un processus. Tous les threads d'un même processus partagent le même espace mémoire (variables globales, tas), mais possèdent chacun leur propre **pile d'exécution** (stack).

En Python, le module `threading` fournit l'API haut niveau pour créer et gérer des threads.

In [ ]:
import threading

print(f"Thread principal : {threading.current_thread().name}")
print(f"Nombre de threads actifs : {threading.active_count()}")

Le thread principal (`MainThread`) est celui qui exécute votre programme. Tout thread supplémentaire que vous créez s'exécute **en parallèle** (du point de vue de l'OS) dans le même processus.

---

## 2. Créer un thread avec `threading.Thread`

La manière la plus simple est de passer une fonction `target` au constructeur de `Thread`.

In [ ]:
import threading
import time

def tache(nom: str, duree: float) -> None:
    print(f"[{nom}] Début")
    time.sleep(duree)
    print(f"[{nom}] Fin après {duree}s")

t = threading.Thread(target=tache, args=("worker", 1.0))
t.start()
print("Le thread principal continue...")
t.join()  # attend la fin du thread
print("Thread worker terminé.")

**Points clés :**

- `start()` lance le thread (il ne faut **jamais** appeler `run()` directement).
- `join()` bloque le thread appelant jusqu'à la fin du thread cible.
- Sans `join()`, le programme principal pourrait se terminer avant le thread.

### Lancer plusieurs threads en parallèle

In [ ]:
import threading
import time

def tache(nom: str, duree: float) -> None:
    print(f"[{nom}] Début")
    time.sleep(duree)
    print(f"[{nom}] Fin")

threads = []
for i in range(5):
    t = threading.Thread(target=tache, args=(f"T-{i}", 0.5), name=f"Worker-{i}")
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("Tous les threads sont terminés.")

### Thread par héritage

On peut aussi sous-classer `Thread` et redéfinir `run()`. C'est moins courant mais utile pour des threads complexes.

In [ ]:
import threading
import time

class MonThread(threading.Thread):
    def __init__(self, n: int) -> None:
        super().__init__(name=f"Calcul-{n}")
        self.n = n
        self.resultat: int | None = None

    def run(self) -> None:
        self.resultat = sum(range(self.n))

t = MonThread(1_000_000)
t.start()
t.join()
print(f"Résultat : {t.resultat}")

---

## 3. Threads démons vs non-démons

Un **thread démon** (`daemon=True`) est automatiquement tué quand le thread principal se termine. Un thread non-démon empêche la sortie du programme tant qu'il n'a pas fini.

In [ ]:
import threading
import time

def surveillance() -> None:
    while True:
        print("[daemon] je surveille...")
        time.sleep(0.3)

d = threading.Thread(target=surveillance, daemon=True)
d.start()
time.sleep(1)
print("Le programme principal s'arrête — le daemon sera tué.")

**Règle pratique :** utilisez `daemon=True` pour les tâches de fond (monitoring, heartbeats) qui n'ont pas besoin de se terminer proprement. Sinon, préférez un thread normal avec un mécanisme d'arrêt explicite (comme un `Event`).

In [ ]:
import threading

t = threading.Thread(target=lambda: None)
print(f"daemon par défaut : {t.daemon}")

t2 = threading.Thread(target=lambda: None, daemon=True)
print(f"daemon explicite : {t2.daemon}")

---

## 4. Le GIL — Global Interpreter Lock

Le **GIL** est un verrou global de l'interpréteur CPython qui empêche plusieurs threads d'exécuter du bytecode Python **simultanément**. C'est la raison principale pour laquelle le multithreading Python n'accélère **pas** les tâches CPU-bound.

### Pourquoi le GIL existe-t-il ?

- Il simplifie la gestion mémoire (comptage de références) de CPython.
- Il rend les extensions C thread-safe sans effort.
- Il existe depuis les débuts de Python (1992).

### Conséquence

| Type de tâche | Threading efficace ? | Alternative |
|---|---|---|
| **I/O-bound** (réseau, fichiers, sleep) | Oui | `asyncio` |
| **CPU-bound** (calcul pur Python) | Non | `multiprocessing`, C extensions |

In [ ]:
import threading
import time

def cpu_bound(n: int) -> int:
    """Tâche CPU-bound : somme naïve."""
    total = 0
    for i in range(n):
        total += i
    return total

N = 5_000_000

# Séquentiel
start = time.perf_counter()
cpu_bound(N)
cpu_bound(N)
seq = time.perf_counter() - start
print(f"Séquentiel : {seq:.3f}s")

In [ ]:
import threading
import time

def cpu_bound(n: int) -> int:
    total = 0
    for i in range(n):
        total += i
    return total

N = 5_000_000

# Avec threads
start = time.perf_counter()
t1 = threading.Thread(target=cpu_bound, args=(N,))
t2 = threading.Thread(target=cpu_bound, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()
threaded = time.perf_counter() - start
print(f"Avec threads : {threaded:.3f}s")
print(f"Le threading n'accélère pas le CPU-bound (GIL) !")

### Quand le GIL est-il relâché ?

Le GIL est **relâché** pendant les opérations I/O (réseau, fichiers, `time.sleep`) et dans certaines extensions C (NumPy, par exemple). C'est pourquoi le threading est efficace pour les tâches I/O-bound.

In [ ]:
import threading
import time

def io_bound(nom: str) -> None:
    """Simule une tâche I/O (requête réseau, lecture fichier)."""
    time.sleep(1)  # GIL relâché pendant sleep

# Séquentiel : 5 x 1s = 5s
start = time.perf_counter()
for i in range(5):
    io_bound(f"T-{i}")
seq = time.perf_counter() - start
print(f"Séquentiel : {seq:.2f}s")

# Threading : ~1s (parallèle)
start = time.perf_counter()
threads = [threading.Thread(target=io_bound, args=(f"T-{i}",)) for i in range(5)]
for t in threads: t.start()
for t in threads: t.join()
par = time.perf_counter() - start
print(f"Avec threads : {par:.2f}s")
print(f"Accélération : {seq / par:.1f}x")

---

## 5. `Lock` — exclusion mutuelle

Quand plusieurs threads accèdent à la même donnée en écriture, il faut protéger la section critique avec un **verrou** (`Lock`). Sans verrou, les opérations non-atomiques créent des **race conditions**.

In [ ]:
import threading

# Exemple de race condition (sans verrou)
compteur = 0

def incrementer(n: int) -> None:
    global compteur
    for _ in range(n):
        compteur += 1  # NON atomique : read-modify-write

threads = [threading.Thread(target=incrementer, args=(100_000,)) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Attendu : 1_000_000, obtenu : {compteur}")
print(f"Race condition : {'oui' if compteur != 1_000_000 else 'non'}")

In [ ]:
import threading

# Avec un Lock : résultat correct
compteur = 0
verrou = threading.Lock()

def incrementer_safe(n: int) -> None:
    global compteur
    for _ in range(n):
        with verrou:  # acquire + release automatique
            compteur += 1

threads = [threading.Thread(target=incrementer_safe, args=(100_000,)) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Attendu : 1_000_000, obtenu : {compteur}")

**Bonne pratique :** toujours utiliser un `Lock` dans un bloc `with` pour garantir la libération même en cas d'exception.

### Timeout sur `acquire()`

On peut passer un `timeout` à `acquire()` pour éviter un blocage infini.

In [ ]:
import threading

verrou = threading.Lock()
verrou.acquire()  # pris par le thread principal

# Tenter de l'acquérir avec timeout
reussi = verrou.acquire(timeout=0.5)
print(f"Acquisition réussie : {reussi}")  # False après 0.5s

verrou.release()

---

## 6. `RLock` — verrou réentrant

Un `RLock` (reentrant lock) peut être acquis **plusieurs fois** par le même thread. Chaque `acquire()` doit être suivi d'un `release()`. C'est utile quand une fonction verrouillée appelle une autre fonction qui prend le même verrou.

In [ ]:
import threading

rverrou = threading.RLock()

def externe() -> None:
    with rverrou:
        print("externe : verrou acquis")
        interne()  # appelle une fonction qui prend le même verrou

def interne() -> None:
    with rverrou:  # OK avec RLock, deadlock avec Lock !
        print("interne : verrou acquis (réentrant)")

externe()

In [ ]:
import threading

# Avec un Lock normal, la même situation créerait un deadlock
lock = threading.Lock()

lock.acquire()
# lock.acquire()  # ← décommenter = deadlock (le thread attend indéfiniment)
lock.release()
print("Lock normal : on ne peut pas acquérir deux fois dans le même thread.")

---

## 7. `Event` — signalisation entre threads

Un `Event` est un drapeau booléen partagé. Un thread peut attendre (`wait()`) que l'événement soit activé (`set()`) par un autre thread. C'est le mécanisme le plus simple pour **signaler** entre threads.

In [ ]:
import threading
import time

pret = threading.Event()

def serveur() -> None:
    print("[serveur] Démarrage...")
    time.sleep(1)  # simule l'initialisation
    print("[serveur] Prêt !")
    pret.set()  # signale que le serveur est prêt

def client() -> None:
    print("[client] En attente du serveur...")
    pret.wait()  # bloque jusqu'au set()
    print("[client] Serveur détecté, je travaille.")

t_srv = threading.Thread(target=serveur)
t_cli = threading.Thread(target=client)
t_cli.start()
t_srv.start()
t_srv.join()
t_cli.join()

### Utiliser `Event` pour arrêter proprement un thread

In [ ]:
import threading
import time

stop = threading.Event()

def worker() -> None:
    while not stop.is_set():
        print("[worker] en cours...")
        stop.wait(timeout=0.3)  # sleep interruptible
    print("[worker] arrêt propre.")

t = threading.Thread(target=worker)
t.start()
time.sleep(1)
stop.set()  # demande l'arrêt
t.join()
print("Thread arrêté proprement.")

---

## 8. `local()` — données thread-local

`threading.local()` crée un objet dont les attributs sont **spécifiques à chaque thread**. Chaque thread voit sa propre copie.

In [ ]:
import threading

donnees = threading.local()

def worker(nom: str) -> None:
    donnees.nom = nom  # chaque thread a son propre .nom
    print(f"[{threading.current_thread().name}] donnees.nom = {donnees.nom}")

threads = [
    threading.Thread(target=worker, args=(f"Thread-{i}",), name=f"T-{i}")
    for i in range(3)
]
for t in threads: t.start()
for t in threads: t.join()

`threading.local()` est souvent utilisé pour stocker des connexions de base de données ou des sessions HTTP **par thread**, évitant les problèmes de partage.

---

## 9. Race conditions et débogage

### 9.1. Anatomie d'une race condition

Une race condition survient quand le résultat dépend de l'ordre d'exécution des threads, qui est **non déterministe**.

In [ ]:
import threading

# Exemple classique : retrait concurrent sur un compte
solde = 100

def retirer(montant: int) -> None:
    global solde
    if solde >= montant:  # (1) vérification
        # Un autre thread peut modifier solde ici !
        solde -= montant  # (2) modification
        print(f"Retiré {montant}, solde = {solde}")
    else:
        print(f"Solde insuffisant ({solde})")

t1 = threading.Thread(target=retirer, args=(80,))
t2 = threading.Thread(target=retirer, args=(80,))
t1.start(); t2.start()
t1.join(); t2.join()
print(f"Solde final : {solde} (peut être négatif !)")

### 9.2. La solution correcte

In [ ]:
import threading

solde = 100
verrou_solde = threading.Lock()

def retirer_safe(montant: int) -> None:
    global solde
    with verrou_solde:  # section critique atomique
        if solde >= montant:
            solde -= montant
            print(f"Retiré {montant}, solde = {solde}")
        else:
            print(f"Solde insuffisant ({solde})")

t1 = threading.Thread(target=retirer_safe, args=(80,))
t2 = threading.Thread(target=retirer_safe, args=(80,))
t1.start(); t2.start()
t1.join(); t2.join()
print(f"Solde final : {solde} (toujours >= 0)")

### 9.3. Détecter les deadlocks

Un **deadlock** survient quand deux threads attendent chacun un verrou détenu par l'autre.

In [ ]:
import threading

# Illustration d'un deadlock potentiel (NE PAS exécuter en prod)
lock_a = threading.Lock()
lock_b = threading.Lock()

def thread_1() -> None:
    with lock_a:
        print("T1 a lock_a, attend lock_b...")
        # time.sleep(0.01)  # augmente la probabilité du deadlock
        with lock_b:
            print("T1 a les deux")

def thread_2() -> None:
    with lock_b:  # ordre inversé → deadlock !
        print("T2 a lock_b, attend lock_a...")
        with lock_a:
            print("T2 a les deux")

# Solution : toujours acquérir les locks dans le MÊME ORDRE
def thread_2_safe() -> None:
    with lock_a:  # même ordre que thread_1
        with lock_b:
            print("T2 safe a les deux")

print("Règle d'or : acquérir les locks toujours dans le même ordre.")

### 9.4. `threading.enumerate()` pour le débogage

In [ ]:
import threading

for t in threading.enumerate():
    print(f"  {t.name} (daemon={t.daemon}, alive={t.is_alive()})")

---

## 10. Synthèse

| Concept | Clé |
|---|---|
| `Thread(target=fn)` | Crée un thread ; `.start()` le lance, `.join()` l'attend |
| `daemon=True` | Thread tué quand le programme s'arrête |
| **GIL** | Empêche le parallélisme CPU en CPython ; pas d'impact sur I/O |
| `Lock` | Exclusion mutuelle — toujours avec `with` |
| `RLock` | Verrou réentrant (même thread peut acquérir plusieurs fois) |
| `Event` | Drapeau booléen pour signaler entre threads |
| `local()` | Données privées par thread |
| Race condition | Résultat dépend de l'ordre d'exécution → verrou |
| Deadlock | Deux threads attendent chacun un verrou de l'autre → même ordre |

**Règle simple :**

- I/O-bound → `threading` (ou `asyncio`)
- CPU-bound → `multiprocessing` (notebook suivant)

---

## 11. Exercices

### Exercice 1 — Téléchargement parallèle simulé *(facile)*

Écrire une fonction `telecharger(url: str, delai: float)` qui simule un téléchargement avec `time.sleep(delai)`. Lancer 5 téléchargements en parallèle avec des threads et mesurer le temps total.

Vérifier que le temps total est proche du plus long délai (pas de la somme).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Threading_et_gil", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import threading
import time

def telecharger(url: str, delai: float) -> None:
    print(f"Téléchargement de {url}...")
    time.sleep(delai)
    print(f"{url} terminé.")

urls = [
    ("page1.html", 0.5),
    ("page2.html", 1.0),
    ("page3.html", 0.3),
    ("image.png", 1.5),
    ("data.json", 0.8),
]

start = time.perf_counter()
threads = [threading.Thread(target=telecharger, args=(url, delai)) for url, delai in urls]
for t in threads:
    t.start()
for t in threads:
    t.join()
elapsed = time.perf_counter() - start
print(f"Temps total : {elapsed:.2f}s (proche de 1.5s, pas de 4.1s)")
```

</details>

### Exercice 2 — Compteur thread-safe *(moyen)*

Écrire une classe `CompteurSafe` avec :

- un attribut `valeur` protégé par un `Lock` ;
- une méthode `incrementer(n: int)` qui incrémente `n` fois ;
- une méthode `obtenir() -> int`.

Lancer 10 threads qui incrémentent chacun 100 000 fois. Vérifier que le résultat est 1 000 000.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Threading_et_gil", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import threading

class CompteurSafe:
    def __init__(self) -> None:
        self._valeur = 0
        self._lock = threading.Lock()

    def incrementer(self, n: int) -> None:
        for _ in range(n):
            with self._lock:
                self._valeur += 1

    def obtenir(self) -> int:
        with self._lock:
            return self._valeur

c = CompteurSafe()
threads = [threading.Thread(target=c.incrementer, args=(100_000,)) for _ in range(10)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"Valeur : {c.obtenir()} (attendu : 1_000_000)")
```

</details>

### Exercice 3 — Arrêt propre avec Event *(moyen)*

Écrire un thread `Moniteur` qui affiche l'heure toutes les 0.5 secondes. Le thread principal attend 3 secondes puis demande l'arrêt via un `Event`. Vérifier que le thread s'arrête proprement (pas de `daemon`).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Threading_et_gil", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import threading
import time
from datetime import datetime

arret = threading.Event()

def moniteur() -> None:
    while not arret.is_set():
        print(f"[Moniteur] {datetime.now().isoformat(timespec='seconds')}")
        arret.wait(timeout=0.5)  # sleep interruptible
    print("[Moniteur] Arrêt propre.")

t = threading.Thread(target=moniteur)
t.start()
time.sleep(3)
arret.set()
t.join()
print("Programme terminé.")
```

</details>

### Exercice 4 — Pool de workers maison *(difficile)*

Implémenter un pool de threads **sans** utiliser `concurrent.futures`. Votre classe `ThreadPool` doit :

1. Créer `n` threads workers au démarrage.
2. Accepter des tâches via `submit(fn, *args)` (les stocker dans une `queue.Queue`).
3. Les workers prennent les tâches de la queue et les exécutent.
4. `shutdown()` envoie un signal d'arrêt et attend la fin des workers.

Tester en soumettant 20 tâches (sleep 0.1s chacune) sur un pool de 4 workers.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Threading_et_gil", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import threading
import queue
import time

class ThreadPool:
    _SENTINEL = object()

    def __init__(self, n_workers: int) -> None:
        self._queue: queue.Queue = queue.Queue()
        self._workers = [
            threading.Thread(target=self._worker, name=f"Pool-{i}", daemon=True)
            for i in range(n_workers)
        ]
        for w in self._workers:
            w.start()

    def _worker(self) -> None:
        while True:
            item = self._queue.get()
            if item is self._SENTINEL:
                break
            fn, args = item
            try:
                fn(*args)
            except Exception as e:
                print(f"Erreur : {e}")
            finally:
                self._queue.task_done()

    def submit(self, fn, *args) -> None:
        self._queue.put((fn, args))

    def shutdown(self) -> None:
        self._queue.join()  # attend que toutes les tâches soient traitées
        for _ in self._workers:
            self._queue.put(self._SENTINEL)
        for w in self._workers:
            w.join()

def tache(i: int) -> None:
    time.sleep(0.1)
    print(f"Tâche {i} terminée par {threading.current_thread().name}")

start = time.perf_counter()
pool = ThreadPool(4)
for i in range(20):
    pool.submit(tache, i)
pool.shutdown()
elapsed = time.perf_counter() - start
print(f"Temps total : {elapsed:.2f}s (4 workers × 5 rounds × 0.1s ≈ 0.5s)")
```

</details>

### Exercice 5 — Mesurer l'impact du GIL *(difficile / deep dive)*

Écrire un benchmark qui compare :

1. Une tâche CPU-bound exécutée 4 fois séquentiellement.
2. La même tâche répartie sur 4 threads.
3. La même tâche répartie sur 4 processus (`multiprocessing.Process`).

Afficher les temps et l'accélération relative. Conclure sur l'impact du GIL.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Threading_et_gil", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
import threading
import multiprocessing
import time

def cpu_work(n: int = 5_000_000) -> int:
    return sum(range(n))

# 1. Séquentiel
start = time.perf_counter()
for _ in range(4):
    cpu_work()
t_seq = time.perf_counter() - start

# 2. Threading
start = time.perf_counter()
threads = [threading.Thread(target=cpu_work) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
t_thread = time.perf_counter() - start

# 3. Multiprocessing
start = time.perf_counter()
procs = [multiprocessing.Process(target=cpu_work) for _ in range(4)]
for p in procs: p.start()
for p in procs: p.join()
t_proc = time.perf_counter() - start

print(f"Séquentiel   : {t_seq:.3f}s")
print(f"Threading    : {t_thread:.3f}s (×{t_seq/t_thread:.1f})")
print(f"Multiprocess : {t_proc:.3f}s (×{t_seq/t_proc:.1f})")
print()
print("Conclusion : le threading n'accélère pas le CPU-bound (GIL).")
print("Le multiprocessing apporte un vrai parallélisme.")
```

</details>

---

## Ressources

- [docs Python — `threading`](https://docs.python.org/3/library/threading.html)
- [docs Python — GIL](https://docs.python.org/3/glossary.html#term-global-interpreter-lock)
- [RealPython — An Intro to Threading in Python](https://realpython.com/intro-to-python-threading/)
- David Beazley — *Understanding the Python GIL* (PyCon 2010)
- [PEP 703 — Making the GIL Optional](https://peps.python.org/pep-0703/) (notebook 05 de ce module)